# F1 Pit Stop — Preprocessing Pipeline

**Input** : `s3://{bucket}/raw/train.csv`  
**Output** : `s3://{bucket}/processed/` — four files ready for modelling

```
processed/
├── X_train_lgbm.parquet   ← LightGBM features
├── X_test_lgbm.parquet
├── X_train_lr.parquet     ← Logistic Regression features (StandardScaler applied)
├── X_test_lr.parquet
├── y_train.csv
└── y_test.csv
```

**10 features selected for interpretability.**  
Both models use identical columns — LR additionally gets StandardScaler.

## 0. Imports

In [18]:
import io
import os
import warnings
warnings.filterwarnings('ignore')

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, TargetEncoder

load_dotenv()

SEED     = 42
TARGET   = 'PitNextLap'
BUCKET   = os.environ.get('S3_BUCKET', 'f1-pitstop-bucket')
RAW_PFX  = 'raw/'
PROC_PFX = 'processed/'

s3 = boto3.client('s3')
print('Setup complete.')

Setup complete.


## 1. Load from S3 & Train/Test Split

In [19]:
def read_csv_from_s3(bucket: str, key: str) -> pd.DataFrame:
    """Download a CSV from S3 into a DataFrame."""
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))


df = read_csv_from_s3(BUCKET, RAW_PFX + 'train.csv').set_index('id')

train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df[TARGET]
)

print(f'Train : {train.shape[0]:,} rows')
print(f'Test  : {test.shape[0]:,} rows')
print(f'Class ratio — train: {train[TARGET].mean():.3f}  test: {test[TARGET].mean():.3f}')

Train : 351,312 rows
Test  : 87,828 rows
Class ratio — train: 0.199  test: 0.199


## 2. Identify Feature Types

In [20]:
y_train = train[TARGET].copy()
y_test  = test[TARGET].copy()
X_train = train.drop(columns=TARGET).copy()
X_test  = test.drop(columns=TARGET).copy()

cat_cols = X_train.select_dtypes(include=['object', 'bool']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object', 'bool']).columns.tolist()

print(f'Numerical   ({len(num_cols)}): {num_cols}')
print(f'Categorical ({len(cat_cols)}): {cat_cols}')

Numerical   (11): ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']
Categorical (3): ['Driver', 'Compound', 'Race']


## 3. Feature Engineering

All transformations are **fit on `X_train` only**, then applied to both splits.

### 3.1 Ratio Feature

In [21]:
def add_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    TyreLife / LapNumber: relative tyre age vs race progress.
    Ratio close to 1.0 means the car has been on this tyre since lap 1.
    """
    df = df.copy()
    df['TyreLife_LapNumber_ratio'] = (
        df['TyreLife'] / df['LapNumber'].clip(lower=1)
    ).astype('float32')
    return df


X_train = add_ratio_features(X_train)
X_test  = add_ratio_features(X_test)
print('Added: TyreLife_LapNumber_ratio')

Added: TyreLife_LapNumber_ratio


### 3.2 Target Encoding for Compound and Race

In [22]:
# Encode Compound and Race as their historical pit rate.
# e.g. Compound_te = 0.18 means soft tyres pit in 18% of laps on average.
# Fit on X_train / y_train only — no leakage into X_test.

te_cols = ['Compound', 'Race']

for c in te_cols:
    X_train[c] = X_train[c].astype('category')
    X_test[c]  = X_test[c].astype('category')

te = TargetEncoder(
    random_state=SEED,
    shuffle=True,
    cv=5,
    smooth='auto',
    target_type='binary'
)

for col in te_cols:
    X_train[f'{col}_te'] = te.fit_transform(X_train[[col]], y_train).astype('float32')
    X_test[f'{col}_te']  = te.transform(X_test[[col]]).astype('float32')

print('Added: Compound_te, Race_te')
print('Sample Compound_te values:')
print(X_train[['Compound', 'Compound_te']].drop_duplicates().sort_values('Compound_te'))

Added: Compound_te, Race_te
Sample Compound_te values:
            Compound  Compound_te
id                               
131893           WET     0.024861
252792           WET     0.026746
387176           WET     0.028540
390051           WET     0.030625
79482            WET     0.031783
384981        MEDIUM     0.101077
106120        MEDIUM     0.101375
300100        MEDIUM     0.101427
152811        MEDIUM     0.101672
355275        MEDIUM     0.101756
41642   INTERMEDIATE     0.150261
302200  INTERMEDIATE     0.151356
386198  INTERMEDIATE     0.151667
406882  INTERMEDIATE     0.153822
316586  INTERMEDIATE     0.154071
91573           SOFT     0.192208
337914          SOFT     0.192954
41601           SOFT     0.193098
919             SOFT     0.195666
2987            SOFT     0.195708
371208          HARD     0.325310
385981          HARD     0.326749
133949          HARD     0.326905
4692            HARD     0.327127
405686          HARD     0.327268


## 4. Select Final Feature Set

10 features chosen for **interpretability** — each has a clear F1 domain meaning.

| Feature | Origin | Plain-English meaning |
|---------|--------|-----------------------|
| `TyreLife` | Original | How many laps this tyre has been on |
| `Cumulative_Degradation` | Original | Total wear accumulated on current tyre |
| `LapNumber` | Original | Current lap number |
| `RaceProgress` | Original | Race completion ratio (0 → 1) |
| `LapTime_Delta` | Original | Change in lap time vs previous lap |
| `Stint` | Original | Which stint of the race (1st, 2nd, …) |
| `Position` | Original | Current race position |
| `TyreLife_LapNumber_ratio` | **Engineered** | Relative tyre age — ratio close to 1 means no stop yet |
| `Compound_te` | **Engineered** | Historical pit rate for this tyre type |
| `Race_te` | **Engineered** | Historical pit rate at this circuit |

In [23]:
SELECTED_FEATURES = [
    # Original features (7)
    'TyreLife',               # tyre age in laps
    'Cumulative_Degradation', # total tyre wear
    'LapNumber',              # current lap
    'RaceProgress',           # 0-1 race completion
    'LapTime_Delta',          # lap time change vs prev lap
    'Stint',                  # stint number
    'Position',               # race position

    # Engineered features (3)
    'TyreLife_LapNumber_ratio', # relative tyre age
    'Compound_te',              # pit rate by tyre compound
    'Race_te',                  # pit rate by circuit
]

X_train = X_train[SELECTED_FEATURES].copy()
X_test  = X_test[SELECTED_FEATURES].copy()

print(f'Final feature count: {len(SELECTED_FEATURES)}')
print(f'X_train: {X_train.shape}   X_test: {X_test.shape}')
print()
print('Original  (7):', SELECTED_FEATURES[:7])
print('Engineered(3):', SELECTED_FEATURES[7:])

Final feature count: 10
X_train: (351312, 10)   X_test: (87828, 10)

Original  (7): ['TyreLife', 'Cumulative_Degradation', 'LapNumber', 'RaceProgress', 'LapTime_Delta', 'Stint', 'Position']
Engineered(3): ['TyreLife_LapNumber_ratio', 'Compound_te', 'Race_te']


## 5. Build Model-Specific Versions

| Version | Extra step | For |
|---------|-----------|-----|
| LGBM | none | LightGBM |
| LR   | StandardScaler | Logistic Regression |

In [24]:
# LGBM version — no scaling needed
X_train_lgbm = X_train.copy()
X_test_lgbm  = X_test.copy()

print(f'LGBM — X_train: {X_train_lgbm.shape}  X_test: {X_test_lgbm.shape}')

LGBM — X_train: (351312, 10)  X_test: (87828, 10)


In [27]:
# LR version — StandardScaler on the same 10 columns
scaler = StandardScaler()

X_train_lr = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_lr = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print(f'LR — X_train: {X_train_lr.shape}  X_test: {X_test_lr.shape}')

LR — X_train: (351312, 10)  X_test: (87828, 10)


## 6. Upload Processed Data to S3

In [28]:
def upload_parquet(df: pd.DataFrame, bucket: str, key: str) -> None:
    """Serialise DataFrame to Parquet in-memory and upload to S3."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=True)
    buf.seek(0)
    s3.upload_fileobj(buf, bucket, key)
    print(f's3://{bucket}/{key}  ({df.shape[0]:,} rows × {df.shape[1]} cols)')


def upload_csv(series: pd.Series, bucket: str, key: str) -> None:
    """Serialise a Series to CSV in-memory and upload to S3."""
    buf = io.StringIO()
    series.to_csv(buf, index=True, header=True)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue().encode())
    print(f's3://{bucket}/{key}  ({len(series):,} rows)')


print('Uploading to S3 ...')
upload_parquet(X_train_lgbm, BUCKET, PROC_PFX + 'X_train_lgbm.parquet')
upload_parquet(X_test_lgbm,  BUCKET, PROC_PFX + 'X_test_lgbm.parquet')
upload_parquet(X_train_lr,   BUCKET, PROC_PFX + 'X_train_lr.parquet')
upload_parquet(X_test_lr,    BUCKET, PROC_PFX + 'X_test_lr.parquet')
upload_csv(y_train, BUCKET, PROC_PFX + 'y_train.csv')
upload_csv(y_test,  BUCKET, PROC_PFX + 'y_test.csv')
print('\nAll files uploaded successfully.')

Uploading to S3 ...
s3://f1-pitstop-bucket/processed/X_train_lgbm.parquet  (351,312 rows × 10 cols)
s3://f1-pitstop-bucket/processed/X_test_lgbm.parquet  (87,828 rows × 10 cols)
s3://f1-pitstop-bucket/processed/X_train_lr.parquet  (351,312 rows × 10 cols)
s3://f1-pitstop-bucket/processed/X_test_lr.parquet  (87,828 rows × 10 cols)
s3://f1-pitstop-bucket/processed/y_train.csv  (351,312 rows)
s3://f1-pitstop-bucket/processed/y_test.csv  (87,828 rows)

All files uploaded successfully.


## 7. Verify Uploads

In [29]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PROC_PFX)
print(f'Files in s3://{BUCKET}/{PROC_PFX}')
print('-' * 55)
for obj in response.get('Contents', []):
    size_kb = obj['Size'] / 1024
    print(f"  {obj['Key']:<45}  {size_kb:>8.1f} KB")

Files in s3://f1-pitstop-bucket/processed/
-------------------------------------------------------
  processed/                                          0.0 KB
  processed/X_test_lgbm.parquet                    1973.1 KB
  processed/X_test_lr.parquet                      2124.5 KB
  processed/X_train_lgbm.parquet                   6870.3 KB
  processed/X_train_lr.parquet                     7177.5 KB
  processed/y_test.csv                             1007.5 KB
  processed/y_train.csv                            4030.2 KB
